In [25]:
from neo4j import GraphDatabase
import pandas as pd

# Neo4j 连接配置（替换为你的实际信息）
NEO4J_URI = "neo4j+s://32ed3653.databases.neo4j.io"  # Aura 用户用 "neo4j+s://xxx.databases.neo4j.io"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "FEvyWPh7BVdppt5CeW-nBpyYRNd2HhYurBlfrXdoiDo"

# 创建驱动实例
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# 测试连接的函数
def test_connection():
    try:
        with driver.session() as session:
            result = session.run("RETURN 1 AS test")
            return result.single()["test"] == 1
    except Exception as e:
        print(f"连接失败: {e}")
        return False

# 执行测试
if test_connection():
    print("✅ Neo4j 连接成功")
else:
    print("❌ 连接失败，请检查配置")

✅ Neo4j 连接成功


In [3]:
def run_query(query, params=None):
    with driver.session() as session:
        result = session.run(query, params)
        return pd.DataFrame([dict(record) for record in result])

In [4]:



# 查询所有节点类型（无LIMIT限制）
query = """
MATCH (n)
RETURN labels(n)[0] AS node_type, count(*) AS count
ORDER BY count DESC
"""  # 已移除 LIMIT 10

df = run_query(query)
df.head(13)


,node_type,count
0,images,23824
1,item,23231
2,title,13719
3,measurement,12449
4,description,3091
5,material,2357
6,period,1697
7,date,1673
8,artist,869
9,donor,649


In [5]:
query = """
MATCH (n:item)-[:`博物馆`]->() LIMIT 25
RETURN n.title AS title
"""
df = run_query(query)
print(df)
#只需要把neo4j的查询cypher语言写道query中，运行函数即可查询

                                                title
0                           Embroidered Silk Fan Case
1   Embroidered Burgundy Silk Damask Jacket for a ...
2         Embroidered Silk Dragon Roundel from a Robe
3   Front Panel of a Rank Badge for a Fifth Rank C...
4                        Album of Landscape Paintings
5                     Couched Gold Apricot Silk Panel
6   Embroidered Cream Silk Bed Cover for Export To...
7       Embroidered Blue Silk Dragon Robe for a Child
8         Print from the Qianlong Emperor’s Conquests
9                          Silk Tapestry Weave Jacket
10                       Red Silk Brocade Chair Cover
11                  Silk Tapestry Weave Table Frontal
12  Silk Twill Uncut Facings for a Woman's Coat or...
13                Embroidered Silk Jacket for a Woman
14                   Rocks and Stream in Bamboo Grove
15                Album Leaf from Album of Landscapes
16                  Embroidered Terracotta Silk Panel
17  Back Panel of a Rank Bad

In [6]:
query = """
MATCH (a)-[r]->(b)
RETURN DISTINCT 
  labels(a)[0] AS start_label,
  type(r) AS relation_type,
  labels(b)[0] AS end_label
"""
df = run_query(query)
print(df.to_string(index=False))


start_label relation_type   end_label
       item            材质    material
       item            类别    category
       item            文化     culture
       item          制作地点       place
       item          对象类型        type
       item          捐赠来源       donor
       item          材料术语    glossary
       item            尺寸 measurement
       item            时期      period
       item           博物馆      museum
       item            图片      images
       item            名称       title
       item          制作时间        date
       item            描述 description
       item           创作者      artist
       item            题跋 inscription


In [18]:
query = """
MATCH (n)
UNWIND labels(n) AS label
UNWIND keys(n) AS field
RETURN DISTINCT label AS 节点类型, field AS 字段名称
ORDER BY label, field
"""

# 执行查询
df = run_query(query)

# 设置 pandas 显示参数
pd.set_option('display.colheader_justify', 'center')  # 列标题居中
pd.set_option('display.width', 1000)  # 设置显示宽度
pd.set_option('display.max_columns', None)  # 显示所有列

# 输出表格（无 index，去掉不合法参数 border）
print(df.to_string(index=False))


    节点类型     字段名称 
     artist  value
   category  value
    culture  value
       date  value
description  value
      donor  value
   glossary  value
     images  value
inscription  value
       item itemId
       item  title
       item    url
   material  value
measurement  value
     museum  value
     period  value
      place  value
      title  value
       type  value


In [28]:
query = """
MATCH (source)-[r]->(target)
WITH DISTINCT
  type(r) AS 关系类型,
  labels(source)[0] AS 源节点类型,
  labels(target)[0] AS 目标节点类型
RETURN
  关系类型,
  源节点类型 + "->" + 目标节点类型 AS 方向,
  源节点类型,
  目标节点类型
ORDER BY 关系类型
"""

df = run_query(query)

def print_table(df):
    col_widths = [max(len(str(x)) for x in [col] + df[col].astype(str).tolist()) for col in df.columns]

    def print_separator():
        print("+" + "+".join("-" * (w + 2) for w in col_widths) + "+")

    def print_row(row):
        print("| " + " | ".join(str(val).ljust(w) for val, w in zip(row, col_widths)) + " |")

    print_separator()
    print_row(df.columns)
    print_separator()

    for _, row in df.iterrows():
        print_row(row)

    print_separator()

print_table(df)


+------+-------------------+-------+-------------+
| 关系类型 | 方向                | 源节点类型 | 目标节点类型      |
+------+-------------------+-------+-------------+
| 创作者  | item->artist      | item  | artist      |
| 制作地点 | item->place       | item  | place       |
| 制作时间 | item->date        | item  | date        |
| 博物馆  | item->museum      | item  | museum      |
| 名称   | item->title       | item  | title       |
| 图片   | item->images      | item  | images      |
| 对象类型 | item->type        | item  | type        |
| 尺寸   | item->measurement | item  | measurement |
| 捐赠来源 | item->donor       | item  | donor       |
| 描述   | item->description | item  | description |
| 文化   | item->culture     | item  | culture     |
| 时期   | item->period      | item  | period      |
| 材料术语 | item->glossary    | item  | glossary    |
| 材质   | item->material    | item  | material    |
| 类别   | item->category    | item  | category    |
| 题跋   | item->inscription | item  | inscription |
+------+-------------------+---